# Imports

In [1]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import pandas as pd
import wandb
import copy
from tabulate import tabulate
import tqdm
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# from diffi.utils import *
from sklearn.ensemble import IsolationForest
from sklearn.metrics import f1_score, average_precision_score

Setting up W&B

In [2]:
wandb.login()

wandb: Currently logged in as: sanson-sebastiano-00 (sanson-sebastiano-00-universita-di-padova) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

Setting a seed value for reproducibility

In [3]:
np.random.seed(0)

# Helper functions

## Logging 

In [4]:
def log_feature_importance(feature_importances, threshold_type, og_model: bool):
    """
    Log feature importance plot to Weights & Biases.

    Args:
        feature_importances (np.ndarray): Array of feature importances.
        threshold_type (str): Type of thresholding used for feature selection.
        og_model (bool): Flag indicating if the model is original or pruned.
    """

    sorted_indices = np.argsort(feature_importances)[::-1]
    fi_std = np.std(feature_importances)

    plt.figure(figsize=(10, 5))
    plt.grid(True, axis='y', linestyle='--', alpha=0.7, zorder=0)
    plt.bar(range(len(sorted_indices)), feature_importances[sorted_indices], 
            yerr=fi_std, zorder=3)
    plt.xticks(range(len(sorted_indices)), sorted_indices)
    plt.xlabel('Feature Index')
    plt.ylabel('Feature Importance')
    plt.ylim(bottom=0)
    if og_model:
        plt.title('Feature Importance - Original Model')
        wandb.log({f"threshold_type_{threshold_type}/feature_importance_original": wandb.Image(plt)})
    else:
        plt.title('Feature Importance - Pruned Model')
        wandb.log({f"threshold_type_{threshold_type}/feature_importance_pruned": wandb.Image(plt)})
    plt.close()

In [5]:
def log_fi_heatmap(fis_in, fis_out, threshold_type, seed, is_og_model: bool):
    """
    Log feature importance heatmaps to wandb with inliers and outliers side-by-side.
    
    Args:
        fis_in (list): List of feature importance matrices for inliers, one per forest.
        fis_out (list): List of feature importance matrices for outliers, one per forest.
        threshold_type (str): Type of thresholding used for feature selection.
    """
    num_forests = len(fis_in)
    assert len(fis_out) == num_forests, "Number of forests doesn't match between inliers and outliers"
    
    for i in range(num_forests):
        # Check if the matrices are empty or have zero dimensions
        if (isinstance(fis_in[i], np.ndarray) and (fis_in[i].size == 0 or fis_in[i].shape[0] == 0)) or \
           (isinstance(fis_out[i], np.ndarray) and (fis_out[i].size == 0 or fis_out[i].shape[0] == 0)):
            print(f"Warning: Empty feature importance matrix for forest {i+1}. Skipping visualization.")
            wandb.log({f"threshold_type_{threshold_type}/seed_{seed}/warning": 
                      f"Empty feature importance matrix for forest {i+1} - visualization skipped"})
            continue

        fig, axes = plt.subplots(1, 2, figsize=(20, 8))
        
        # Inliers heatmap
        sns.heatmap(fis_in[i], cmap='viridis', cbar=True, vmin=0, vmax=0.16, ax=axes[0])
        if is_og_model:
            axes[0].set_title(f'Original Model Feature Importance - Inliers (Forest {i+1})', fontsize=14)
        else:
            axes[0].set_title(f'Pruned Model Feature Importance - Inliers (Forest {i+1})', fontsize=14)
        axes[0].set_xlabel('Feature Index')
        axes[0].set_ylabel('Tree Index')
        
        # Outliers heatmap 
        sns.heatmap(fis_out[i], cmap='viridis', cbar=True, vmin=0, vmax=0.16, ax=axes[1])
        if is_og_model:
            axes[1].set_title(f'Original Model Feature Importance - Outliers (Forest {i+1})', fontsize=14)
        else:   
            axes[1].set_title(f'Pruned Model Feature Importance - Outliers (Forest {i+1})', fontsize=14)
        axes[1].set_xlabel('Feature Index')
        axes[1].set_ylabel('Tree Index')
        
        plt.tight_layout()
        
        if is_og_model:
            wandb.log({f"threshold_type_{threshold_type}/seed_{seed}/feature_importance_heatmap_original": wandb.Image(fig)})
        else:
            wandb.log({f"threshold_type_{threshold_type}/seed_{seed}/feature_importance_heatmap_pruned": wandb.Image(fig)})
        
        plt.close(fig)

In [6]:
def log_fi_diff(og_fi, pruned_fi, threshold_type):
    """
    Log the difference in features importance between the original and pruned model.
    
    Args:
        og_fi (np.ndarray): Feature importance of the original model.
        pruned_fi (np.ndarray): Feature importance of the pruned model.
        threshold_type (str): Type of thresholding used for feature selection.
    """
    fi_diff = pruned_fi - og_fi
    colors = ['green' if val > 0 else 'red' for val in fi_diff]

    plt.figure(figsize=(10, 5))
    plt.grid(True, axis='y', linestyle='--', alpha=0.7, zorder=0)
    plt.bar(range(len(fi_diff)), fi_diff, color=colors, zorder=3)
    plt.xticks(range(len(fi_diff)), range(len(fi_diff)))
    plt.xlabel('Feature Index')
    plt.ylabel('Change in Feature Importance (Pruned - Original)')
    plt.title('Difference in Feature Importance After Pruning')

    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='green', edgecolor='black', label='Increase after pruning'),
        Patch(facecolor='red', edgecolor='black', label='Decrease after pruning')
    ]
    plt.legend(handles=legend_elements, loc='upper right')

    wandb.log({f"threshold_type_{threshold_type}/feature_importance_difference": wandb.Image(plt)})
    plt.close()

In [7]:
def log_estimators_summary(pruned_num_trees, og_num_trees, threshold_type, seed):
    """
    Log the number of estimators in each Isolation Forest to wandb.

    Args:
        iforests (list): List of Isolation Forest models.
        seed (int): Random seed used for the models.
        og_num_trees (int): Original number of trees in the unpruned model.
        threshold_type (str): Type of thresholding used for feature selection.
    """

    # # Log per-forest statistics
    # for i, iforest in enumerate(iforests):
    #     forest_count = len(iforest.estimators_)
    #     forest_index = i + 1  # Use 1-based indexing for readability
        
    #     wandb.log({
    #         f"threshold_type_{threshold_type}/seed_{seed}/forest_{forest_index}/num_estimators": forest_count,
    #         # Percentage of trees retained compared to original 
    #         f"threshold_type_{threshold_type}/seed_{seed}/forest_{forest_index}/retention_rate": forest_count / og_num_trees,
    #         f"threshold_type_{threshold_type}/seed_{seed}/forest_{forest_index}/mean_estimators": np.mean(forest_count),
    #         f"threshold_type_{threshold_type}/seed_{seed}/forest_{forest_index}/min_estimators": np.min(forest_count),
    #         f"threshold_type_{threshold_type}/seed_{seed}/forest_{forest_index}/max_estimators": np.max(forest_count),
    #     })

    # wandb.log({
    #     f"threshold_type_{threshold_type}/seed_{seed}/overall_total_estimators": np.sum([len(iforest.estimators_) for iforest in iforests]),
    # })

    # Log summary table
    data = [[i+1, pruned_num_trees[i]] for i in range(len(pruned_num_trees))]
    columns = ["Forest Index", "Number of Estimators"]
    wandb.log({f"threshold_type_{threshold_type}/seed_{seed}/estimators_table": wandb.Table(data=data, columns=columns)})

In [8]:
def log_features_summary(features_importance, selected_features, seed, threshold_type):
    """
    Log a summary of feature importances and selected meaningful features to wandb.
    Args:
        features_importance (np.ndarray): Array of feature importances.
        selected_features (list): List of indices of selected meaningful features for each forest.
        threshold_type (str): Type of thresholding used for feature selection.
    """

    for f in range(len(features_importance)):
        num_all_features = len(features_importance[f])
        num_meaningful_features = len(selected_features[f])
        # plot bar chart for each forest, highlighting selected features
        plt.figure(figsize=(10, 5))
        plt.grid(True, axis='y', linestyle='--', alpha=0.7, zorder=0)
        bars = plt.bar(range(len(features_importance[f])), features_importance[f], zorder=3)
        plt.xticks(range(len(features_importance[f])), range(len(features_importance[f])))
        plt.xlabel('Feature Index')
        plt.ylabel('Feature Importance')
        plt.ylim(bottom=0)
        for idx in selected_features[f]:
            bars[idx].set_color('orange')
        plt.legend(['Selected Features'], loc='upper right')
        plt.text(0.981, 0.88, f'Selected {num_meaningful_features}/{num_all_features} meaningful features', 
                 horizontalalignment='right', verticalalignment='top', transform=plt.gca().transAxes,
                 bbox=dict(facecolor='white', alpha=0.8, edgecolor='black'))
        plt.title(f'Feature Importance - Forest {f+1} (Seed {seed})')
        wandb.log({f"threshold_type_{threshold_type}/seed_{seed}/meaningful_features_selected": wandb.Image(plt)})
        plt.close()

In [9]:
def log_depth_vs_usage(num_forests, num_trees, unique_usages, avg_outliers_depth_mat, threshold_type, seed, isTrainingSet: bool):
    """
    Log bar plots comparing unique usages and average outliers depth for each tree in each forest.
    Args:
        num_forests (int): Number of forests.
        num_trees (int): Number of trees in each forest.
        unique_usages (list): List of lists containing unique usages for each tree in each forest.
        avg_outliers_depth_mat (list): List of lists containing average outliers depth for each tree in each forest.
        threshold_type (str): Type of thresholding used for feature selection.
        seed (int): Random seed used for the models.
    """ 
    outlier_label = "Avg Training Outliers Depth" if isTrainingSet else "Avg Prediction Outliers Depth"

    for f in range(num_forests):
        plt.figure(figsize=(15, 7))
        width = 0.35  # the width of the bars
        x = np.arange(num_trees)  # the label locations

        plt.bar(x - width/2, unique_usages[f], width, label='Unique Usages', color='blue', alpha=0.7)
        plt.bar(x + width/2, avg_outliers_depth_mat[f], width, label=outlier_label, color='orange', alpha=0.7)   
        plt.xlabel('Tree Index')
        plt.ylabel('Value')
        plt.title(f'Comparison of Unique Usages and {outlier_label} (Seed {seed}, Forest {f+1})')
        plt.legend()
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.tight_layout()
        wandb.log({f"threshold_type_{threshold_type}/seed_{seed}/unique_usages_vs_avg_outliers_depth": wandb.Image(plt)})
        plt.close()

## Usage feature counter

In [10]:
def get_feature_usage(used_features, unique_features):
    """
    Calculating the feature usage in each tree for each forest.

    Args: 
        used_features (list): 
            - each element represents a forest and it is a list
                - each element of a forest is a tree and contains a list of features used in that tree
        unique_features (list): list of unique features used in the dataset.
    Returns:
        usage (np.ndarray): shape (num_forests, num_trees, num_features) 
            Each element is the count of how many times a feature is used in a tree of a forest.
        normalized_usage (np.ndarray): shape (num_forests, num_trees, num_features)
            Each element is the normalized count of how many times a feature is used in a tree of a forest. Normalization
            wrt to the total splits used in the tree. 
    """

    num_forests = len(used_features)
    num_trees = len(used_features[0])
    num_features = len(unique_features)

    usage = np.zeros((num_forests, num_trees, num_features), dtype=float)
    normalized_usage = np.zeros((num_forests, num_trees, num_features), dtype=float)

    # Iterate over each forest
    for f in range(num_forests):
        # Iterate over each tree in the forest
        for t in range(num_trees):
            internal_nodes_counter = 0
            # Iterate over each feature used in the tree
            for feature in used_features[f][t]:
                if feature in unique_features:
                    feature_index = unique_features.index(feature)
                    usage[f, t, feature_index] += 1
                if feature != -2:  # not a leaf node
                    internal_nodes_counter += 1
            
            # Normalize the usage by the number of internal nodes in the tree
            if internal_nodes_counter > 0:
                normalized_usage[f, t] = usage[f, t] / internal_nodes_counter
            else:
                raise ValueError(f"Tree {t} in forest {f} has no internal nodes, cannot normalize usage.")

    return usage, normalized_usage

## Meaningful feature selection methods

In [11]:
def elbow_features_selection(feature_importances):
    """
    Selecting most important features based on the elbow method.

    Args:
        feature_importances (np.ndarray): Array of shape (num_forests, num_features)
            containing the feature importances for each forest.

    Returns:
        selected_features (list): List of lists, where each inner list contains the indices of the selected features for each forest.
        threshold_gaps (np.ndarray): Array of shape (num_forests,) containing the maximum gap thresholds for each forest.
        gaps (list): List of lists, where each inner list contains the gaps between consecutive feature importances for each forest.
    """

    n_forests, n_features = feature_importances.shape

    # Get the indexes of the features sorted by importance
    sorted_indices = np.argsort(feature_importances, axis=1)[:, ::-1]

    # Sort the feature importances based on the sorted indices
    sorted_importances = np.take_along_axis(feature_importances, sorted_indices, axis=1)

    selected_features, gaps = [], []
    threshold_gaps = np.zeros(n_forests, dtype=float)

    if n_features < 2: # Not enough features to compute gaps
        for f in range(n_forests):
            selected_idx = sorted_indices[f, :].tolist()
            selected_features.append(selected_idx)
            gaps.append([])
    else:
        # Calculate gaps between consecutive feature importances
        gaps_batch = sorted_importances[:, :-1] - sorted_importances[:, 1:]

        # Get the maximum gap for each forest
        max_gap_indices = np.argmax(gaps_batch + 1e-12 * np.random.rand(*gaps_batch.shape), axis=1)
        threshold_gaps = np.take_along_axis(gaps_batch, max_gap_indices[:, np.newaxis], axis=1).squeeze()

        # Select features based on the maximum gap
        for f in range(n_forests):
            num_to_select = max_gap_indices[f] + 1
            selected_idx_for_forest = sorted_indices[f, :num_to_select].tolist()
            selected_features.append(selected_idx_for_forest)
            gaps.append(gaps_batch[f, :].tolist())

    return selected_features, threshold_gaps, gaps

## Selection of trees to be removed

In [12]:
def get_unique_usage_per_tree(normalized_usages, feature_importances, selected_features):
    """
    Compute the unique usage of selected features for each tree in each forest as weighted sum of features usage weighted by their importance.

    Args:
        normalized_usages (np.ndarray): Array of shape (num_forests, num_trees, num_features)
            containing the normalized feature usage for each tree in each forest.
        feature_importances (np.ndarray): Array of shape (num_forests, num_features)
            containing the feature importances for each forest.
        selected_features (list): List of lists, where each inner list contains the indices of the selected features for each forest.

    Returns:
        unique_usages (np.ndarray): Array of shape (num_forests, num_trees) containing the weighted usage for each tree in each forest.
    """

    n_forests, n_trees, n_features = normalized_usages.shape
    unique_usages = np.zeros((n_forests, n_trees), dtype=float)

    for f in range(n_forests):
        forest_selected_features = set(selected_features[f])
        forest_total_importance = np.sum(feature_importances[f, :])
        assert forest_total_importance > 0, f"Forest {f} has zero total importance, cannot compute unique usage."

        for t in range(n_trees):
            current_tree_weighted_usage = 0.0
            current_tree_weight_sum = 0.0

            for feature in range(n_features):
                if feature in forest_selected_features:
                    current_tree_weighted_usage += normalized_usages[f, t, feature] * feature_importances[f, feature]
                    current_tree_weight_sum += feature_importances[f, feature]

            assert current_tree_weight_sum > 0, f"Tree {t} in forest {f} has zero total importance, cannot compute unique usage."
            unique_usages[f, t] = current_tree_weighted_usage / current_tree_weight_sum
            
    return unique_usages

In [13]:
def get_unique_avg_usage_per_forest(normalized_usages, selected_features):
    """
    Compute the average usage of selected features across all trees in each forest.

    Args:
        normalized_usages (np.ndarray): Array of shape (num_forests, num_trees, num_features)
            containing the normalized feature usage for each tree in each forest.
        selected_features (list): List of lists, where each inner list contains the indices of the selected features for each forest.

    Returns:
        unique_avg_usage (np.ndarray): Array of shape (num_forests, num_features) containing the average usage of selected feature for each forest.
    """

    n_forests, _, n_features = normalized_usages.shape
    unique_avg_usage = np.zeros((n_forests, n_features), dtype=float)

    for f in range(n_forests):
        forest_selected_features = set(selected_features[f])

        for feature in range(n_features):
            if feature in forest_selected_features:
                unique_avg_usage[f, feature] = np.mean(normalized_usages[f, :, feature])

    return unique_avg_usage

def majority_voting(normalized_usages, selected_features, feature_importances):
    """ 
    Apply majority voting to compute weighted voting score 
    Args:
        normalized_usages (np.ndarray): Array of shape (num_forests, num_trees, num_features)
            containing the normalized feature usage for each tree in each forest.
        selected_features (list): List of lists, where each inner list contains the indices of the selected features for each forest.
        feature_importances (np.ndarray): Array of shape (num_forests, num_features)
            containing the feature importances for each forest.
    Returns:
        voting_scores (np.ndarray): Array of shape (num_forests, num_trees) containing the voting score for each tree in each forest.
    """
    
    n_forests, n_trees, n_features = normalized_usages.shape
    binary_scores = np.zeros((n_forests, n_trees, n_features), dtype=int)
    voting_scores = np.zeros((n_forests, n_trees), dtype=float)

    unique_avg_usage = get_unique_avg_usage_per_forest(normalized_usages, selected_features)

    for f in range(n_forests):
        forest_selected_features = set(selected_features[f])
        forest_total_importance = np.sum(feature_importances[f, :])
        assert forest_total_importance > 0, f"Forest {f} has zero total importance, cannot compute voting score."

        for t in range(n_trees):
            current_tree_weight_sum = 0.0

            for feature in range(n_features):
                if feature in forest_selected_features:
                    if normalized_usages[f, t, feature] > unique_avg_usage[f, feature]:
                        binary_scores[f, t, feature] = 1
                    current_tree_weight_sum += feature_importances[f, feature]

            assert current_tree_weight_sum > 0, f"Tree {t} in forest {f} has zero total importance, cannot compute voting score."
            voting_scores[f, t] = np.sum(np.multiply(binary_scores[f, t, :], feature_importances[f, :])) 

    return voting_scores

In [14]:
def remove_trees(iforests, unique_usages, thresholds):
    """
    For each forest, remove trees that have a feature usage below a certain threshold.

    Args:
        iforests (list): List of forests, each containing a list of trees.
        unique_usages (np.ndarray): Array of shape (num_forests, num_trees, num_features) 
            containing the usage of each feature in each tree.
        thresholds (list): List of thresholds for each forest to determine which trees to remove.

    Returns:
        pruned_forests (list): List of pruned forests with trees removed based on the thresholds.
    """
    pruned_forests = copy.deepcopy(iforests)

    n_forests = len(iforests)
    keep_mask = unique_usages > thresholds[:, np.newaxis]

    for f in range(n_forests):
        # Get the index of the forest to prune
        current_forest = pruned_forests[f]
        current_mask = keep_mask[f]

        # Ensure at least one tree remains
        if not np.any(current_mask):
            # Keep the tree with highest usage
            best_tree_idx = np.argmax(unique_usages[f])
            current_mask[best_tree_idx] = True
            print(f"Warning: All trees would be pruned in forest {f}. Keeping the best tree.")

        og_estimators = current_forest.estimators_
        og_features = current_forest.estimators_features_

        # Filter and update the estimators and features based on the mask
        pruned_forests[f].estimators_ = [est for idx, est in enumerate(og_estimators) if current_mask[idx]]
        pruned_forests[f].estimators_features_ = [feat for idx, feat in enumerate(og_features) if current_mask[idx]]

        # Update also the internal attributes of the forest
        og_decision_paths = current_forest._decision_path_lengths
        pruned_forests[f]._decision_path_lengths = [path for idx, path in enumerate(og_decision_paths) if current_mask[idx]]

        og_avg_paths = current_forest._average_path_length_per_tree
        pruned_forests[f]._average_path_length_per_tree = [avg_path for idx, avg_path in enumerate(og_avg_paths) if current_mask[idx]]
        pruned_forests[f].n_estimators_ = len(pruned_forests[f].estimators_)
        
    return pruned_forests


In [15]:
def random_trees_removal(iforests, num_trees_to_remove, seed):
    """
    For each forest, remove trees that have a feature usage below a certain threshold.

    Args:
        iforests (list): List of forests, each containing a list of trees.
        unique_usages (np.ndarray): Array of shape (num_forests, num_trees, num_features) 
            containing the usage of each feature in each tree.
        thresholds (list): List of thresholds for each forest to determine which trees to remove.

    Returns:
        pruned_forests (list): List of pruned forests with trees removed based on the thresholds.
    """
    pruned_forests = copy.deepcopy(iforests)

    n_forests = len(iforests)

    for f in range(n_forests):
        # Get the index of the forest to prune
        current_forest = pruned_forests[f]
        n_trees = len(current_forest.estimators_)

        if num_trees_to_remove >= n_trees:
            raise ValueError(f"Cannot remove {num_trees_to_remove} trees from forest {f} with only {n_trees} trees.")
        
        # Mask of trees to keep (True) and remove (False)
        keep_mask = np.ones(n_trees, dtype=bool)
        rnd = np.random.RandomState(seed=seed + f)  
        indices_to_remove = rnd.choice(n_trees, num_trees_to_remove, replace=False)
        keep_mask[indices_to_remove] = False

        og_estimators = current_forest.estimators_
        og_features = current_forest.estimators_features_

        # Filter and update the estimators and features based on the mask
        pruned_forests[f].estimators_ = [est for idx, est in enumerate(og_estimators) if keep_mask[idx]]
        pruned_forests[f].estimators_features_ = [feat for idx, feat in enumerate(og_features) if keep_mask[idx]]

        # Update also the internal attributes of the forest
        og_decision_paths = current_forest._decision_path_lengths
        pruned_forests[f]._decision_path_lengths = [path for idx, path in enumerate(og_decision_paths) if keep_mask[idx]]

        og_avg_paths = current_forest._average_path_length_per_tree
        pruned_forests[f]._average_path_length_per_tree = [avg_path for idx, avg_path in enumerate(og_avg_paths) if keep_mask[idx]]
        pruned_forests[f].n_estimators_ = len(pruned_forests[f].estimators_)
        
    return pruned_forests


In [16]:
def get_depth_stats(iforests, X_train, y):
    """
    Get depth statistics for each forest.

    Args:
        iforests (list): List of Isolation Forest models.
        X_train (np.ndarray): Training data.
        y (np.ndarray): Training or predicted labels
    Returns:
        outliers_depth (np.ndarray): Matrix of shape (num_forests, num_trees, num_outliers)
            containing the depth statistics for each outlier in each tree of each forest.
    """
    num_forests = len(iforests)
    num_trees = len(iforests[0].estimators_)
    outliers_depth = np.zeros((num_forests, num_trees), dtype=float)

    # For each forest
    for i in range(len(iforests)):
        current_iforest = iforests[i]
        
        # Extract current predictions
        current_y = y[i] if isinstance(y, list) or (isinstance(y, np.ndarray) and len(y.shape) > 1) else y
        
        # Get outlier indices
        outliers = np.argwhere(current_y == 1).flatten()
        num_outliers = len(outliers)
        assert num_outliers > 0, f"No outliers found in forest {i}"

        # For each tree
        for j in range(num_trees):
            current_estimator = current_iforest.estimators_[j]
            
            # Get leaf nodes for outliers
            outlier_samples = X_train[outliers]
            on_leaf = current_estimator.apply(outlier_samples)
            
            # Get node depths
            node_depths = current_estimator.tree_.compute_node_depths()
            
            depth_values = []
            # For each outlier
            for o in range(num_outliers):
                leaf_idx = on_leaf[o]
                if leaf_idx < len(node_depths) and current_estimator.tree_.children_left[leaf_idx] == -1:
                    depth = node_depths[leaf_idx]
                    if depth > 0:
                        depth_values.append(1/depth)
        
            if depth_values:
                outliers_depth[i, j] = np.mean(depth_values)
    
    return outliers_depth

In [17]:
def post_pruned_evaluation(X_test:np.ndarray, y_test:np.ndarray, pruned_if:IsolationForest):
    """
    Evaluate the pruned Isolation Forest models on the test set.
    Args:
        X_test (np.ndarray): Test data.
        y_test (np.ndarray): True labels for the test data.
        pruned_if (list): List of pruned Isolation Forest models.
    Returns:
        f1s (np.ndarray): Array of F1-scores for each pruned model.
        avps (np.ndarray): Array of Average Precision scores for each pruned model.
    """
    
    f1s, avps = [], []

    for f in range(len(pruned_if)):
        # get predictions
        y_pred = pruned_if[f].predict(X_test)
        y_pred = np.where(y_pred == -1, 1, 0)  # map -1 to 1 (anomalies), 1 to 0 (inliers)
        anomaly_scores = 0.5 * (-pruned_if[f].decision_function(X_test) + 1)
        # compute performance metrics
        f1 = f1_score(y_test, y_pred)
        avg_precision = average_precision_score(y_test, anomaly_scores)

        f1s.append(f1)
        avps.append(avg_precision)

    return np.asarray(f1s), np.asarray(avps)

In [18]:
def diffi_ranks(X_train, X_test, y_test, seed, n_iters, contamination: float | str = 'auto', num_trees=100, max_samples=256): 
    
    models = []
    used_features = []
    f1s, avps = [], []
    fis, fis_out, fis_in = [], [], []
    ys_pred_train = []

    for k in range(n_iters): 

        iforest = IsolationForest(n_estimators=num_trees, max_samples=max_samples, 
                                  contamination=contamination, random_state = seed + k) 
        
        iforest.fit(X_train) 

        # lo stesso vale per gli anomaly scores, vanno tra -1 e 1 (-1=inlier) ma li vogliamo 
        # tra 0 e 1 dove 1 = anomalous

        # -1 for anomalies, 1 for inliers are returned
        y_pred_train = iforest.predict(X_train)     # for depth computation
        y_pred_test = iforest.predict(X_test)       # for performance metrics
        # mapping: -1 -> 1 (anomalies), 1 -> 0 (inliers)
        # the true labels are 0 for inliers and 1 for anomalies
        y_pred_train = np.where(y_pred_train == -1, 1, 0)
        y_pred_test = np.where(y_pred_test == -1, 1, 0)
        
        anomaly_scores = 0.5 *(-iforest.decision_function(X_test) + 1) 

        # compute performance metrics
        f1 = f1_score(y_test, y_pred_test)
        avg_precision = average_precision_score(y_test, anomaly_scores)

        # compute feature importance
        # fi, _, fi_outliers_per_tree, fi_inliers_per_tree = interp.diffi_ib_per_tree(iforest, X_train)
        # fi_outliers_per_tree è una lista con num_trees elementi. Ogni elemento
        # ha 13 feature importance, una per ogni feature

        models.append(iforest)
        used_features.append([tree.tree_.feature for tree in iforest.estimators_])

        f1s.append(f1)
        avps.append(avg_precision)
        # fis.append(fi)
        # fis_out.append(fi_outliers_per_tree)
        # fis_in.append(fi_inliers_per_tree)
        ys_pred_train.append(y_pred_train)

    return np.asarray(f1s), np.asarray(avps), np.asarray(fis), models, used_features, fis_out, fis_in, np.asarray(ys_pred_train)


# Experiments

## Parameters

In [19]:
num_trees = 100
max_samples = 256  
num_forests = 10
test_size = 0.2   # 0.2
seeds = np.arange(30).tolist()
majority_voting_threshold = 0.5  # Threshold for majority voting
num_random_runs = 30    # Number of random runs for comparison

In [20]:
threshold_type = 'random'      # 'random', 'mean', 'majority_voting', 'percentile' 
selection_method = 'elbow'      # 'elbow', 'soft_threshold'
percentile = 80             # for percentile thresholding
num_trees_to_remove = 50    # for random removal
depth_flag = False               # Whether to compute depth based removal or not
areTrueOutliers = False         # Whether to compute depth based on true or predicted outliers

## Experiment loop

In [21]:
normal_data = pd.read_csv('tep_dataset/TEP_FaultFree_Training_subsample_70_3.csv')
anomalous_data = pd.read_csv('tep_dataset/TEP_Faulty_Training_subsample_70_3_removedfirst20.csv')

faulty_data, datasets = [], []
idx_fault = [0, 1, 5, 7, 11, 12, 17] 

for fault_num in idx_fault:
    fault_data = anomalous_data[anomalous_data['faultNumber'] == fault_num + 1].reset_index(drop=True)
    faulty_data.append(fault_data)
    datasets.append(pd.concat([normal_data, fault_data]).reset_index(drop=True))

# summary_df = pd.DataFrame({
#     'Fault Number': [idx_fault[i] + 1 for i in range(len(datasets))],
#     'Dataset Shape': [datasets[i].shape for i in range(len(datasets))],
#     'Normal Samples': [len(normal_data)] * len(datasets),
#     'Fault Samples': [len(datasets[i]) - len(normal_data) for i in range(len(datasets))]
# })

# display(summary_df)

In [22]:
assert threshold_type in ['random', 'mean', 'majority_voting', 'percentile'], "Invalid threshold_type"
assert selection_method in ['elbow', 'soft_threshold'], "Invalid selection_method"
# assert depth_flag == True and threshold_type != 'random', "Depth based removal can be applied only with non-random thresholding"
# assert depth_flag == True and threshold_type != 'majority_voting', "Majority voting thresholding does not support depth based removal"

for i, dataset in tqdm.tqdm(enumerate(datasets), desc="Loading datasets", total=len(datasets)):
    # Initialize results list to store the results of each dataset
    results = []

    f1_over_seeds = np.zeros((len(seeds), num_forests))
    avg_precision_over_seeds = np.zeros((len(seeds), num_forests))

    # Split features and target
    X = datasets[i].iloc[:, 3:-1].values 
    y = datasets[i]['Target'].values

    # Get the unique features used in the dataset
    all_features = range(X.shape[1])

    for seed in tqdm.tqdm(seeds, desc=f"Processing dataset with N. Fault: {idx_fault[i]+1}", leave=False):

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=seed, stratify=y) if test_size > 0 else (X, X, y, y)
        contamination = faulty_data[i].shape[0] / dataset.shape[0]
        X_train, y_train = shuffle(X_train, y_train, random_state=seed)

        # Init Weights & Biases run
        run = wandb.init(
            project="if_optimization",
            name=f"diffi_tep_seed_{seed}",
            config={
                "seed": seed,
                "dataset": f"Dataset_Fault_{idx_fault[i]+1}",
                "num_forests": num_forests,
                "num_trees": num_trees,
                "contamination": contamination,
                "max_samples": max_samples,
                "test_size": test_size,
                "selection_method": selection_method if (threshold_type != 'random' and depth_flag == False) else "N/A",          # for random removal
                "thresholds": threshold_type + (f"_depth_based" if depth_flag and threshold_type != 'random' else ""),  # indicate if depth based or not    
                "percentile_value": percentile if threshold_type == 'percentile' else "N/A",            # for percentile thresholding
                "num_trees_to_remove": num_trees_to_remove if threshold_type == 'random' else "N/A",    # for random removal
                "depth_based_on": "true_outliers" if areTrueOutliers and depth_flag else ("predicted_outliers" if depth_flag else "N/A"),  # indicate if depth based on true or predicted outliers
            },
        )

        f1s, avg_precisions, fi_og, iforests, used_features, fis_out, fis_in, y_pred_train = diffi_ranks(
            X_train,
            X_test,
            y_test,
            seed,
            n_iters=num_forests,
            contamination=contamination,
            num_trees=num_trees,  
        )

        mean_f1 = np.mean(f1s)
        mean_avg_precision = np.mean(avg_precisions)
        mean_fi_og = np.mean(fi_og, axis=0)

        # log_feature_importance(mean_fi_og, threshold_type, og_model=True)
        # log_fi_heatmap(fis_in, fis_out, threshold_type, seed, is_og_model=True)

        # ???
        f1_over_seeds[seed, :] = f1s
        avg_precision_over_seeds[seed, :] = avg_precisions

        pruned_iforests =  []

        f1s_pruned, avg_precisions_pruned, fi_pruned, fis_out_pruned, fis_in_pruned, removed_trees = [], [], [], [], [], []

        if threshold_type == 'random':
            partial_n_removed_trees, partial_f1s, partial_avg_precisions, partial_fis = [], [], [], []
            # repeat the random removal `num_random_runs` times
            for s in range(num_random_runs):
                pruned_iforests = random_trees_removal(iforests, num_trees_to_remove, seed=s)
                f1s_pruned, avg_precisions_pruned = post_pruned_evaluation(
                    X_test, 
                    y_test, 
                    pruned_iforests
                )
                partial_n_removed_trees.append([len(pruned_iforests[i].estimators_) for i in range(len(pruned_iforests))])
                partial_f1s.append(f1s_pruned)
                partial_avg_precisions.append(avg_precisions_pruned)
                partial_fis.append(fi_pruned)
            
            # average the results over the runs
            removed_trees = np.mean(np.array(partial_n_removed_trees), axis=0)
            f1s_pruned = np.mean(np.array(partial_f1s), axis=0)
            avg_precisions_pruned = np.mean(np.array(partial_avg_precisions), axis=0)
            fi_pruned = np.mean(np.array(partial_fis), axis=0)
            
        else:   # threshold_type in ['mean', 'percentile', 'majority_voting']
            if depth_flag:
                # get depth stats for outliers
                if areTrueOutliers:
                    outliers_depth_mat = get_depth_stats(iforests, X_train, y_train)  
                else:
                    outliers_depth_mat = get_depth_stats(iforests, X_train, y_pred_train)   
                    
                avg_outliers_depth_mat = outliers_depth_mat  # shape (num_forests, num_trees)  
            else: 
                # feature selection
                selected_features = []
                if selection_method == 'elbow':
                    selected_features, gap_thresholds, gaps = elbow_features_selection(fi_og)
                else:   # 'soft_threshold'
                    selected_features = [all_features]*num_forests
                # log_features_summary(fi_og, selected_features, seed, threshold_type)
                
                # extract only the feature importances of the selected features
                unique_fi_og = np.zeros((num_forests, fi_og.shape[1]), dtype=float)
                for f in range(num_forests):
                    forest_selected_features = set(selected_features[f])
                    
                    for sf in forest_selected_features:
                        unique_fi_og[f, sf] = fi_og[f, sf]
                # normalize the unique feature importances per forest
                normalized_unique_fi_og = np.divide(unique_fi_og, np.sum(unique_fi_og, axis=1, keepdims=True))
                # get feature usage
                usages, normalized_usages = get_feature_usage(used_features, all_features)

                unique_usages = get_unique_usage_per_tree(normalized_usages, normalized_unique_fi_og, selected_features)

            # WARNING: when majority voting is used, this log is redundant
            # log_depth_vs_usage(num_forests, num_trees, unique_usages, avg_outliers_depth_mat, threshold_type, seed, isTrainingSet=False)

            thresholds = np.zeros(num_forests, dtype=float)

            if threshold_type == 'mean':

                if depth_flag:
                    for f in range(num_forests):
                        thresholds[f] = np.mean(avg_outliers_depth_mat[f, :])
                    pruned_iforests = remove_trees(iforests, avg_outliers_depth_mat, thresholds)
                else:
                    for f in range(num_forests):
                        thresholds[f] = np.mean(unique_usages[f, :])
                    pruned_iforests = remove_trees(iforests, unique_usages, thresholds)

            elif threshold_type == 'percentile':

                if depth_flag:
                    for f in range(num_forests):
                        thresholds[f] = np.percentile(avg_outliers_depth_mat[f, :], percentile)
                    pruned_iforests = remove_trees(iforests, avg_outliers_depth_mat, thresholds)
                else:
                    for f in range(num_forests):
                        thresholds[f] = np.percentile(unique_usages[f, :], percentile)
                    pruned_iforests = remove_trees(iforests, unique_usages, thresholds)

            else:   # 'majority_voting'
                voting_scores = majority_voting(normalized_usages, selected_features, normalized_unique_fi_og)

                # log_depth_vs_usage(num_forests, num_trees, voting_scores, avg_outliers_depth_mat, threshold_type, seed, isTrainingSet=False)
                
                pruned_iforests = remove_trees(iforests, voting_scores, np.array([majority_voting_threshold]*num_forests))

            removed_trees = [len(pruned_iforests[i].estimators_) for i in range(len(pruned_iforests))]

            f1s_pruned, avg_precisions_pruned = post_pruned_evaluation(
                X_test, 
                y_test, 
                pruned_iforests
            )

        log_estimators_summary(removed_trees, run.config.num_trees, threshold_type, seed)
            
        mean_f1_pruned = np.mean(f1s_pruned)
        mean_avg_precision_pruned = np.mean(avg_precisions_pruned)
        mean_fi_pruned = np.mean(fi_pruned, axis=0)

        # log_feature_importance(mean_fi_pruned, threshold_type, og_model=False)
        # log_fi_heatmap(fis_in_pruned, fis_out_pruned, threshold_type, seed, is_og_model=False)
        # log_fi_diff(mean_fi_og, mean_fi_pruned, threshold_type)

        results.append([
            seed,
            f"{mean_f1:.4f}",
            f"{mean_f1_pruned:.4f}",
            f"{mean_avg_precision:.4f}",
            f"{mean_avg_precision_pruned:.4f}",
        ])

    headers = ["Seed", "F1 Score", "F1 Score Pruned", "Avg Precision", "Avg Precision Pruned"]
    # table = tabulate(results, headers, tablefmt="github")
    # print(f"\n### Results for Dataset with Fault {idx_fault[i]+1} ###")
    # print(table)

    # log performance metrics to wandb
    table = wandb.Table(data=results, columns=headers)
    wandb.log({f"Dataset_Fault_{idx_fault[i]+1}_results_table": table})

Loading datasets:   0%|          | 0/7 [00:00<?, ?it/s]

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


...

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Loading datasets: 100%|██████████| 7/7 [1:02:27<00:00, 535.37s/it]


In [23]:
run.finish()